# GraphKnows — LLM-Assisted Demonstration

GraphKnows is a **multi-session memory layer** for AI agents. This notebook extends the LLM-free pipeline by layering **DSPy + OpenRouter** on top to extract richer relations and produce human-readable topic summaries for each memory cluster.

| Component | LLM-free baseline | LLM-assisted addition |
|-----------|------------------|----------------------|
| Entity extraction | GLiNER2 (local DeBERTa) | same |
| Relation extraction | GLiNER2 binary triplets | **DSPy** reasons over full chunk text → richer typed relations |
| Graph enrichment | Leiden communities | same + **TOPIC node summarisation** per community |
| Retrieval | DETRetriever (3 arms) | same + `search_topics` arm available in AgenticRetriever |

**What you will see beyond the LLM-free demo:**
- DSPy extracts implicit relations that GLiNER2 missed (e.g. `CEO_OF` inferred from appositive phrasing)
- TOPIC nodes in the Neo4j graph — one amber node per Leiden community, with a GPT-generated label and summary
- `IN_TOPIC` edges linking entities to their topic clusters
- The `ltm_topics` table in PgVector, enabling topic-level semantic search

**Prerequisites:** Everything in `demo_llm_free.ipynb`, plus `OPENROUTER_API_KEY` in `.env`.

In [ ]:
from __future__ import annotations
import asyncio, importlib, json, re, sys, time, uuid
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'graphknows').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
importlib.invalidate_caches()
for _m in [k for k in sys.modules if k == 'graphknows' or k.startswith('graphknows.')]:
    del sys.modules[_m]

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

try:
    from yfiles_jupyter_graphs import GraphWidget
    from ipywidgets import Layout
    from IPython.display import display as _display
    YFILES = True
except Exception:
    GraphWidget = Layout = _display = None
    YFILES = False

from graphknows.config import GraphKnowsSettings
from graphknows.config.builders.stores import (
    build_stm_service, build_promotion_pipeline, build_neo4j_store,
    build_pgvector_store, build_memgraph_store,
)
from graphknows.container import GraphKnowsContainer
from graphknows.memory.models.promotion import PromotionRequest

settings = GraphKnowsSettings()
# Activate DSPy relation extraction during STM ingest
settings.relation_extraction_mode = 'llm_assisted'

LLM_MODEL = 'moonshotai/kimi-k2'  # change to any OpenRouter model

if not (settings.openrouter_api_key or '').strip():
    print('WARNING: OPENROUTER_API_KEY not set — DSPy extraction will be skipped')
else:
    print(f'OpenRouter key: {settings.openrouter_api_key[:8]}...')

_PALETTE = {
    'PERSON': '#4A90D9', 'ORG': '#E8891A', 'ORGANIZATION': '#E8891A',
    'PLACE': '#27AE60', 'LOCATION': '#27AE60', 'GPE': '#27AE60',
    'DATE': '#8E44AD', 'TIME': '#8E44AD', 'EVENT': '#C0392B',
    'PRODUCT': '#16A085', 'CONCEPT': '#7D3C98', 'TOPIC': '#F39C12',
    'ENTITY': '#7F8C8D',
}

# STM service for ingest (uses _run() sync bridge internally — never query ._pg directly)
stm_service = build_stm_service(settings)   # uses llm_assisted relation extractor
flusher     = build_promotion_pipeline(settings)
neo_vis     = build_neo4j_store(settings)

# Dedicated query stores connected in Jupyter's event loop (avoids asyncpg loop conflicts)
pg  = build_pgvector_store(settings)
await pg.connect()
mg  = build_memgraph_store(settings)   # sync bolt driver — no loop conflict

print(f'embed_model              : {settings.embed_model}')
print(f'relation_extraction_mode : {settings.relation_extraction_mode}')
print(f'yfiles                   : {YFILES}')

## Database Setup — Fresh Start

Each demo run purges all three stores so results are never contaminated by a previous run.

| Store | Scope | What is cleared |
|-------|-------|-----------------|
| **Memgraph** | Session-scoped STM graph | All nodes and edges |
| **Neo4j** | Shared LTM graph | All ENTITY and TOPIC nodes |
| **PostgreSQL** | Semantic index | `stm_chunks`, `stm_turns`, `ltm_chunks`, `ltm_turns`, `ltm_topics` |

> Although each run already uses a fresh `SESSION_ID` UUID, Neo4j and the LTM tables are **shared across sessions** — old entities from a previous run would otherwise accumulate in the graph and skew PageRank, community detection, and retrieval results.

In [ ]:
print('Purging all stores for a clean slate...')
t0 = time.monotonic()

# Memgraph — STM structural graph
def _mg_clear():
    with mg._driver.session() as s:
        r = s.run('MATCH (n) DETACH DELETE n RETURN count(n) AS n').single()
        return r['n'] if r else 0

mg_deleted = await asyncio.to_thread(_mg_clear)
print(f'  Memgraph  : {mg_deleted} nodes removed')

# Neo4j — LTM entity graph
def _neo4j_clear():
    with neo_vis._driver.session() as s:
        r = s.run('MATCH (n) DETACH DELETE n RETURN count(n) AS n').single()
        return r['n'] if r else 0

neo_deleted = await asyncio.to_thread(_neo4j_clear)
print(f'  Neo4j     : {neo_deleted} nodes removed')

# PostgreSQL — semantic index tables
async with pg._pool.acquire() as conn:
    for table in ('stm_chunks', 'stm_turns', 'ltm_chunks', 'ltm_turns', 'ltm_topics'):
        n = await conn.fetchval(f'SELECT COUNT(*) FROM {table}')
        await conn.execute(f'DELETE FROM {table}')
        print(f'  {table:<16}: {n} rows removed')

print(f'\nAll stores clean — ready to run  ({round(time.monotonic()-t0, 1)}s)')

## The Benchmark Case: `locomo-smoke-006`

Same case as the LLM-free demo — this lets us compare both pipelines directly on identical input.

The question requires **multi-hop inference across sessions**. Neither GLiNER2 nor DSPy can answer it from any single session. The LLM-assisted advantage is visible in three ways:

1. **Richer relations** in the Neo4j graph — DSPy finds implicit links that GLiNER2 binary extraction misses
2. **TOPIC nodes** — each Leiden community gets a human-readable label and summary
3. Potentially **higher recall** if the path to the answer runs through a typed relation that only DSPy extracted

In [ ]:
FIXTURE = REPO_ROOT / 'evaluation' / 'fixtures' / 'locomo' / 'questions.jsonl'
cases = {}
for line in FIXTURE.read_text().splitlines():
    if line.strip():
        q = json.loads(line)
        cases[q['id']] = q

CASE       = cases['locomo-smoke-006']
SESSION_ID = f'demo-asst-{uuid.uuid4().hex[:8]}'

print(f'Case     : {CASE["id"]}')
print(f'Question : {CASE["question"]}')
print(f'Answer   : {CASE["answer"]}')
print(f'Sessions : {len(CASE.get("sessions", []))}')
print(f'Session  : {SESSION_ID}\n')

for i, sess in enumerate(CASE.get('sessions', []), 1):
    msgs = [m for m in sess.get('messages', []) if m.get('content', '').strip()]
    sid  = sess.get('session_id', f'session-{i}')
    print(f'--- Session {i} [{sid}]  ({len(msgs)} messages) ---')
    for m in msgs[:2]:
        preview = m['content'][:130].replace('\n', ' ')
        print(f'  {m["role"]:>10}: {preview}')
    if len(msgs) > 2:
        print(f'             ... {len(msgs) - 2} more messages')
    print()

## Step 1 — Ingestion into Short-Term Memory (LLM-Assisted)

With `relation_extraction_mode='llm_assisted'`, `STMService` adds a `DSPyRelationExtractor` on top of GLiNER2:

```
text
  → PlainTextParser          chunk at ~400 tokens
  → _SpacySchemaMiner        spaCy entity-type hints
  → GLiNER2EntityExtractor   typed entity spans (same as LLM-free)
  → DSPyRelationExtractor    LLM reasons over the full chunk text
                             → 39 relation types aligned with schema.org
                             → catches implicit relations (appositive, co-reference, …)
  → MemgraphStore            ENTITY -[:<RELATION_TYPE>]→ ENTITY (typed, bi-temporal)
  → PgVectorStore            embed chunk → INSERT INTO stm_chunks
```

DSPy makes one LLM call per chunk. Ingest is slower than LLM-free but produces a denser, more accurate relation graph.

In [ ]:
def _session_text(sess):
    return '\n'.join(
        f"{m['role']}: {m['content']}"
        for m in sess.get('messages', [])
        if m.get('content', '').strip()
    )

print('Ingesting sessions into STM (DSPy relation extraction active)...')
t0 = time.monotonic()
total_chunks, total_entities = 0, 0

for i, sess in enumerate(CASE.get('sessions', []), 1):
    title = sess.get('session_id', f'session-{i}')
    text  = _session_text(sess)
    if text:
        res = stm_service.ingest(
            text=text, session_id=SESSION_ID,
            title=title, metadata={'source': 'locomo', 'session_index': i},
        )
        total_chunks   += res.chunk_count
        total_entities += res.entity_count
        print(f'  Session {i:>2}  {title:<40}  chunks={res.chunk_count:>3}  entities={res.entity_count:>3}')

print(f'\nTotal  {round(time.monotonic()-t0, 1)}s : {total_chunks} chunks, {total_entities} entity mentions')
print('(Slower than LLM-free because DSPy makes one LLM call per chunk)')

## STM Graph: Memgraph — Entity-Relation Structure

Same graph schema as LLM-free. The difference is in the **relation edges**: DSPy produces more typed edges and catches implicit relations that GLiNER2 binary extraction misses.

**What to look for compared to the LLM-free graph:**
- More edges between the same entity pairs
- Relation labels that reflect sentence-level reasoning (`CEO_OF`, `PATRON_OF`, `MANAGES`)
- Coverage of relations stated through appositive phrasing (`"Elara Voss, CEO of Meridian"` → `CEO_OF` edge)

In [ ]:
nodes = await mg.get_session_nodes(SESSION_ID)

def _all_rel_edges():
    with mg._driver.session() as s:
        return [dict(r) for r in s.run(
            'MATCH (a:ENTITY {session_id: $sid})-[r]->(b:ENTITY {session_id: $sid}) '
            'RETURN a.name AS src, b.name AS tgt, type(r) AS rel, '
            'coalesce(r.confidence, 0.8) AS conf',
            sid=SESSION_ID,
        )]

edges = await asyncio.to_thread(_all_rel_edges)
print(f'STM Memgraph: {len(nodes)} entity nodes, {len(edges)} relation edges')
print(f'  (Compare with LLM-free: typically more edges per entity pair)')

seen: set = set()
yf_nodes, yf_edges = [], []
for n in nodes:
    nm = n.get('name', '')
    if nm and nm not in seen:
        seen.add(nm)
        t = (n.get('type') or 'ENTITY').upper()
        yf_nodes.append({
            'id': nm, 'label': nm, 'type': t,
            'properties': {'type': t, 'confidence': round(float(n.get('confidence') or 1), 2)},
        })
for i, e in enumerate(edges):
    src, tgt = e.get('src', ''), e.get('tgt', '')
    for nm in (src, tgt):
        if nm and nm not in seen:
            seen.add(nm)
            yf_nodes.append({'id': nm, 'label': nm, 'type': 'ENTITY', 'properties': {'type': 'ENTITY'}})
    if src and tgt:
        yf_edges.append({'id': f'r{i}', 'start': src, 'end': tgt, 'label': e.get('rel', '?'),
                         'properties': {'confidence': round(float(e.get('conf') or 0.8), 2)}})

if YFILES:
    w = GraphWidget(widget_layout=Layout(height='640px', width='100%'))
    w.directed = True
    w.nodes, w.edges = yf_nodes, yf_edges
    w.set_node_label_mapping(lambda i, n: n['label'])
    w.set_edge_label_mapping(lambda i, e: e.get('label', ''))
    w.set_node_color_mapping(lambda i, n: _PALETTE.get(n.get('type', 'ENTITY'), '#7F8C8D'))
    w.set_node_property_mapping(lambda i, n: n.get('properties', {}))
    w.organic_layout()
    _display(w)
else:
    print(f'  Entities: {chr(44).join(n["label"] for n in yf_nodes[:12])}{"..." if len(yf_nodes) > 12 else ""}')

## STM Index: PgVector — Chunk and Turn Store

Identical table structure to the LLM-free pipeline. DSPy's richer relations are stored as graph edges in Memgraph, **not** in PgVector — the embedding index is unchanged regardless of which extractor ran.

`community_id` and `community_weight` are still `null` at this point — populated during LTM promotion.

In [ ]:
async with pg._pool.acquire() as conn:
    chunks = await conn.fetch(
        'SELECT id, file_id, text, metadata, community_id, community_weight '
        'FROM stm_chunks WHERE session_id = $1 ORDER BY created_at',
        SESSION_ID,
    )
    turns = await conn.fetch(
        'SELECT turn_index, role, text FROM stm_turns '
        'WHERE session_id = $1 ORDER BY turn_index',
        SESSION_ID,
    )

print(f'stm_chunks : {len(chunks)} rows')
print(f'stm_turns  : {len(turns)} rows')
print(f'community_id populated: {any(c["community_id"] is not None for c in chunks)}  ← False until promotion\n')

print('stm_chunks sample:')
print(f'  {"#":<3} {"memory_class":<18} {"community_id":<13} text[:85]')
print('  ' + '─' * 82)
for i, c in enumerate(chunks[:10], 1):
    meta = json.loads(c['metadata']) if c['metadata'] else {}
    mc   = meta.get('memory_class', 'raw_evidence')
    cid  = str(c['community_id']) if c['community_id'] is not None else 'null'
    text = str(c['text'])[:85].replace('\n', ' ')
    print(f'  {i:<3} {mc:<18} {cid:<13} {text}')
if len(chunks) > 10:
    print(f'  ... {len(chunks) - 10} more rows')

## STM Recall — `DETRetriever(scope='stm')`

Same as LLM-free: the `ltm_graph` arm is inactive, so the retriever is limited to vector search + entity keyword matching within this session.

**Expected result: score ≈ 0.** DSPy did extract more relations into Memgraph, but they remain session-scoped. The cross-session inference chain only becomes visible after LTM promotion merges all sessions into one graph.

In [ ]:
async with GraphKnowsContainer.from_settings(settings) as gk:
    stm_ctx = await gk.retriever.retrieve(
        CASE['question'], session_id=SESSION_ID, top_k=5, scope='stm'
    )

expected = str(CASE['answer']).lower()
_tokens  = set(re.findall(r'[a-z]+', expected))
haystack = ' '.join(r.get('text', '') for r in stm_ctx.fused_results).lower()
stm_score = 1.0 if expected in haystack else round(
    len(_tokens & set(re.findall(r'[a-z]+', haystack))) / max(len(_tokens), 1), 3
)

verdict = '✓ FOUND' if stm_score >= 1.0 else '✗ not found — multi-session inference required'
print(f'Question  : {CASE["question"]}')
print(f'Expected  : {CASE["answer"]}')
print(f'STM score : {stm_score:.3f}  ({verdict})\n')

print(f'Top {len(stm_ctx.fused_results)} results:')
for i, r in enumerate(stm_ctx.fused_results, 1):
    sources = r.get('sources', '?')
    text    = str(r.get('text', ''))[:100].replace('\n', ' ')
    print(f'  #{i}  rrf={r.get("rrf_score", 0):.4f}  [{sources}]')
    print(f'       {text}')

## Step 2 — LTM Promotion (LLM-Assisted)

Promotion runs the same seven steps as LLM-free, **plus** step 7 with `topic_mode='llm_assisted'`:

```
1. Read session subgraph from Memgraph
2. EntityResolution   — case-insensitive dedup
3. Neo4jStore         — MERGE into shared LTM graph
4. PgVectorStore      — copy stm_chunks → ltm_chunks
5. GDS: PageRank + betweenness + Leiden community detection
6. PgVectorStore      — propagate community_id + community_weight onto ltm_chunks
7. TopicSummarizer (DSPy)
     For each Leiden community:
       • collect entity names + representative chunk passages
       • LLM call → (title, summary)
       • Neo4jStore writes a TOPIC node
       • ENTITY -[:IN_TOPIC]→ TOPIC edges written
```

The `topics_created` field in the result shows how many community clusters were summarised. Each TOPIC node is visible in Neo4j as an amber node with a GPT-generated label and summary.

In [ ]:
flusher._llm_model = LLM_MODEL  # model for TopicSummarizer

t0 = time.monotonic()
flush = await flusher.promote(PromotionRequest(
    session_id=SESSION_ID,
    run_gds=True,
    clear_stm=False,
    merge_entities=True,
    topic_mode='llm_assisted',  # creates TOPIC nodes via DSPy TopicSummarizer
))

print(f'Promotion complete in {flush.elapsed_s:.1f}s')
print(f'  nodes_written   : {flush.nodes_written}')
print(f'  edges_written   : {flush.edges_written}')
print(f'  merged_entities : {flush.merged_entities}')
print(f'  chunks_promoted : {flush.chunks_promoted}')
print(f'  turns_promoted  : {flush.turns_promoted}')
print(f'  topics_created  : {flush.topics_created}   ← TOPIC nodes written to Neo4j')
if flush.gds_result:
    g = flush.gds_result
    print(f'\n  GDS: algorithm={g.algorithm_used}  communities={g.community_count}', end='')
    if g.modularity:
        print(f'  modularity={g.modularity:.4f}', end='')
    print()
if flush.errors:
    print(f'\n  Errors: {flush.errors}')

## LTM Graph: Neo4j — Cross-Session Graph with TOPIC Nodes

The LTM graph now has three node types:

| Node | Colour | Key Properties |
|------|--------|----------------|
| ENTITY | blue / orange / green / grey | type, pagerank, community_id, confidence |
| TOPIC | **amber (large)** | label (LLM-generated), summary, community_id |

**Edge types:**
- **Typed relations** — `AFFILIATED_WITH`, `WORKS_FOR`, etc. (from GLiNER2 + DSPy)
- **CO_OCCURS** — entities appearing in the same chunk
- **IN_TOPIC** — entity → its TOPIC cluster

**View 1** shows only entities and typed relations — no topic noise.  
**View 2** shows the full graph including TOPIC nodes and IN_TOPIC membership edges.

Click any TOPIC node in the sidebar to read its LLM-generated `summary`.

In [ ]:
async def _neo_q(cypher, **params):
    def _sync():
        with neo_vis._driver.session() as s:
            return [dict(r) for r in s.run(cypher, **params)]
    return await asyncio.to_thread(_sync)

entity_rows = await _neo_q(
    'MATCH (e:ENTITY) '
    'RETURN e.name AS name, coalesce(e.type,"ENTITY") AS type, '
    'coalesce(e.pagerank,0.0) AS pagerank, coalesce(e.community_id,-1) AS community_id, '
    'coalesce(e.confidence,1.0) AS confidence LIMIT 150'
)
rel_rows = await _neo_q(
    'MATCH (a:ENTITY)-[r]->(b:ENTITY) WHERE NOT type(r) IN ["IN_TOPIC","CO_OCCURS"] '
    'RETURN a.name AS src, b.name AS tgt, type(r) AS rel, coalesce(r.weight,1) AS w LIMIT 300'
)
co_rows = await _neo_q(
    'MATCH (a:ENTITY)-[r:CO_OCCURS]->(b:ENTITY) '
    'RETURN a.name AS src, b.name AS tgt, coalesce(r.weight,1) AS w LIMIT 200'
)
topic_rows = await _neo_q(
    'MATCH (t:TOPIC) '
    'RETURN t.id AS id, coalesce(t.label,t.id) AS label, '
    'coalesce(t.summary,"") AS summary, coalesce(t.community_id,-1) AS community_id LIMIT 50'
)
topic_edges = await _neo_q(
    'MATCH (e:ENTITY)-[:IN_TOPIC]->(t:TOPIC) '
    'RETURN e.name AS src, t.id AS tgt LIMIT 300'
)

print(f'LTM Neo4j: {len(entity_rows)} entities, {len(rel_rows)} typed rels, '
      f'{len(co_rows)} CO_OCCURS, {len(topic_rows)} TOPIC nodes, {len(topic_edges)} IN_TOPIC edges')

if topic_rows:
    print('\nTOPIC node summaries:')
    for t in topic_rows:
        summary = (t['summary'] or '')[:120]
        print(f'  [{t["label"]}]  {summary}{"..." if len(t["summary"] or "") > 120 else ""}')

# --- Build node/edge lists ---
seen_n: set = set()
all_entity_nodes, all_topic_nodes = [], []
all_rel_edges, all_co_edges, all_tp_edges = [], [], []

for r in entity_rows:
    nm = r['name']
    if nm and nm not in seen_n:
        seen_n.add(nm)
        pr = float(r['pagerank'] or 0)
        all_entity_nodes.append({
            'id': nm, 'label': nm, 'type': (r['type'] or 'ENTITY').upper(),
            'properties': {'type': r['type'], 'pagerank': round(pr, 4),
                           'community_id': r['community_id'],
                           'confidence': round(float(r['confidence']), 2)},
        })
for t in topic_rows:
    tid = t['id']
    if tid and tid not in seen_n:
        seen_n.add(tid)
        all_topic_nodes.append({
            'id': tid, 'label': t['label'], 'type': 'TOPIC',
            'properties': {'type': 'TOPIC', 'label': t['label'],
                           'summary': (t['summary'] or '')[:300],
                           'community_id': t['community_id']},
        })
for i, r in enumerate(rel_rows):
    src, tgt = r['src'], r['tgt']
    for nm in (src, tgt):
        if nm and nm not in seen_n:
            seen_n.add(nm)
            all_entity_nodes.append({'id': nm, 'label': nm, 'type': 'ENTITY', 'properties': {}})
    if src and tgt:
        all_rel_edges.append({'id': f'r{i}', 'start': src, 'end': tgt, 'label': r['rel'],
                              'properties': {'weight': float(r['w']), 'kind': 'typed'}})
for i, r in enumerate(co_rows):
    src, tgt = r['src'], r['tgt']
    if src and tgt:
        all_co_edges.append({'id': f'c{i}', 'start': src, 'end': tgt, 'label': 'CO_OCCURS',
                             'properties': {'weight': float(r['w']), 'kind': 'co_occurs'}})
for i, r in enumerate(topic_edges):
    src, tgt = r['src'], r['tgt']
    if src and tgt:
        all_tp_edges.append({'id': f't{i}', 'start': src, 'end': tgt, 'label': 'IN_TOPIC',
                             'properties': {'kind': 'in_topic'}})

def _node_colour(i, n):
    t = n.get('type', 'ENTITY')
    return _PALETTE.get(t, '#7F8C8D')

def _node_size(i, n):
    t  = n.get('type', '')
    pr = float((n.get('properties') or {}).get('pagerank') or 0)
    base = 85 if t == 'TOPIC' else 52
    return [base + min(pr * 500, 35)] * 2

def _edge_thick(i, e):
    k = (e.get('properties') or {}).get('kind', 'typed')
    return 0.6 if k == 'co_occurs' else (1.2 if k == 'in_topic' else 2.2)

if YFILES:
    # View 1: entity graph only (no topic nodes)
    print('\n▶ View 1 — Entity graph (typed relations only, no TOPIC nodes)')
    w1 = GraphWidget(widget_layout=Layout(height='620px', width='100%'))
    w1.directed = True
    w1.nodes = all_entity_nodes
    w1.edges = all_rel_edges
    w1.set_node_label_mapping(lambda i, n: n['label'])
    w1.set_edge_label_mapping(lambda i, e: e.get('label', ''))
    w1.set_node_color_mapping(_node_colour)
    w1.set_node_size_mapping(_node_size)
    w1.set_node_property_mapping(lambda i, n: n.get('properties', {}))
    w1.set_edge_thickness_factor_mapping(_edge_thick)
    w1.organic_layout()
    _display(w1)

    # View 2: full graph with TOPIC nodes
    print('\n▶ View 2 — Full LTM graph (entities + TOPIC clusters + CO_OCCURS)')
    w2 = GraphWidget(widget_layout=Layout(height='750px', width='100%'))
    w2.directed = True
    w2.nodes = all_entity_nodes + all_topic_nodes
    w2.edges = all_rel_edges + all_tp_edges + all_co_edges
    w2.set_node_label_mapping(lambda i, n: n['label'])
    w2.set_edge_label_mapping(lambda i, e: e.get('label', ''))
    w2.set_node_color_mapping(_node_colour)
    w2.set_node_size_mapping(_node_size)
    w2.set_node_property_mapping(lambda i, n: n.get('properties', {}))
    w2.set_edge_thickness_factor_mapping(_edge_thick)
    w2.organic_layout()
    _display(w2)
else:
    top = sorted(entity_rows, key=lambda x: -(x.get('pagerank') or 0))[:8]
    print(f'  Top entities by PageRank: {chr(44).join(r["name"] for r in top)}')

## LTM Index: PgVector — After Promotion

Same as LLM-free: `community_id` and `community_weight` are now populated on `ltm_chunks`. The TOPIC summaries live in Neo4j as `TOPIC` nodes **and** in the PgVector `ltm_topics` table for topic-level ANN search.

| Table | Used by | Purpose |
|-------|---------|----------|
| `ltm_chunks` | DETRetriever | Per-chunk semantic search with community boost |
| `ltm_topics` | AgenticRetriever | Topic-level ANN search — `search_topics` arm |

In [ ]:
async with pg._pool.acquire() as conn:
    ltm_chunks = await conn.fetch(
        'SELECT id, file_id, text, metadata, community_id, community_weight '
        'FROM ltm_chunks WHERE session_id = $1 ORDER BY flushed_at',
        SESSION_ID,
    )
    ltm_topics_rows = await conn.fetch(
        'SELECT topic_id, community_id, summary, entity_names FROM ltm_topics LIMIT 10'
    )

comm_ids = [c['community_id'] for c in ltm_chunks if c['community_id'] is not None]
print(f'ltm_chunks : {len(ltm_chunks)} rows')
print(f'community_ids: {len(comm_ids)}/{len(ltm_chunks)} assigned  ({len(set(comm_ids))} clusters)\n')

print('ltm_chunks sample — community metadata populated after GDS:')
print(f'  {"#":<3} {"community_id":<13} {"weight":<9} text[:80]')
print('  ' + '─' * 82)
for i, c in enumerate(ltm_chunks[:10], 1):
    cid  = str(c['community_id']) if c['community_id'] is not None else 'null'
    wt   = f"{c['community_weight']:.4f}" if c['community_weight'] is not None else 'null'
    text = str(c['text'])[:80].replace('\n', ' ')
    print(f'  {i:<3} {cid:<13} {wt:<9} {text}')
if len(ltm_chunks) > 10:
    print(f'  ... {len(ltm_chunks) - 10} more rows')

if ltm_topics_rows:
    print(f'\nltm_topics table: {len(ltm_topics_rows)} rows  (used by AgenticRetriever.search_topics)')
    for t in ltm_topics_rows[:4]:
        entities = list(t['entity_names'] or [])[:4]
        summary  = (t['summary'] or '')[:80]
        print(f'  community={t["community_id"]}  entities={entities}  summary="{summary}"')

## LTM Recall — `DETRetriever(scope='ltm')`

Same three retrieval arms as LLM-free. The LLM-assisted advantage shows up in:

1. **Denser graph** — more typed relations give the `ltm_graph` arm more paths to traverse to the answer chunk
2. **Community boost** — if the answer chunk's entities belong to a well-summarised topic cluster, `community_weight` is higher

> `search_topics` (the fourth arm in `AgenticRetriever`) is not part of `DETRetriever`. To enable topic-level retrieval, use `AgenticRetriever` by setting `OPENROUTER_API_KEY` and removing the `DETRetriever` override in the container.

In [ ]:
async with GraphKnowsContainer.from_settings(settings) as gk:
    ltm_ctx = await gk.retriever.retrieve(
        CASE['question'], session_id=SESSION_ID, top_k=5, scope='ltm'
    )

haystack_ltm = ' '.join(r.get('text', '') for r in ltm_ctx.fused_results).lower()
ltm_score = 1.0 if expected in haystack_ltm else round(
    len(_tokens & set(re.findall(r'[a-z]+', haystack_ltm))) / max(len(_tokens), 1), 3
)

verdict = '✓ FOUND — graph traversal closed the inference chain' if ltm_score >= 1.0 else '✗ not found'
print(f'Question  : {CASE["question"]}')
print(f'Expected  : {CASE["answer"]}')
print(f'LTM score : {ltm_score:.3f}  ({verdict})\n')

print(f'Top {len(ltm_ctx.fused_results)} results:')
for i, r in enumerate(ltm_ctx.fused_results, 1):
    sources  = r.get('sources', '?')
    entities = ', '.join(str(e) for e in r.get('entities', [])[:4])
    text     = str(r.get('text', ''))[:110].replace('\n', ' ')
    print(f'  #{i}  rrf={r.get("rrf_score", 0):.4f}  [{sources}]')
    if entities:
        print(f'       entities: {entities}')
    print(f'       {text}')
    print()

## Score Summary — STM vs LTM

In [ ]:
from IPython.display import Markdown, display

stm_icon = '✓' if stm_score >= 1.0 else ('~' if stm_score > 0 else '✗')
ltm_icon = '✓' if ltm_score >= 1.0 else ('~' if ltm_score > 0 else '✗')

md = f'''
### `{CASE["id"]}` — LLM-Assisted Pipeline Results

**Question:** {CASE["question"]}  
**Expected answer:** `{CASE["answer"]}`

| Stage | Retriever scope | Score | Result | Notes |
|-------|----------------|------:|-------|-------|
| STM (just ingested) | `scope=\'stm\'` | {stm_score:.3f} | {stm_icon} | Multi-session gap |
| LTM (after promotion) | `scope=\'ltm\'` | {ltm_score:.3f} | {ltm_icon} | Graph traversal + vector + community boost |

**What LLM-assistance added:**
- DSPy extracted {len(yf_edges)} relation edges (vs GLiNER2 binary triplets in LLM-free) into Memgraph STM
- {flush.topics_created} TOPIC node(s) created in Neo4j — each a Leiden community with a GPT-generated label and summary
- TOPIC nodes are queryable by `AgenticRetriever.search_topics` for topic-level retrieval
'''
display(Markdown(md))